Chart Analysis
Use Python to retrieve stock market data and analyze price changes.
1. Fetch both daily and hourly stock market data using a suitable API.
Visualize the data in several types of plots.
2. Plot long-term stock data over multiple years.
Create an interactive graph that allows the user to view different time ranges.
3. Start with Apple (AAPL) and plot its stock chart using daily data over several years.
4. Add moving averages to the stock charts, such as:
○ SMA20 and SMA50, or
○ SMA50 and SMA150
These should be overlaid on the main stock price chart.
5. Create a table for five stocks. Use default examples such as FAANG stocks plus
NVIDIA, but allow the user to change them.
For each stock, calculate the historical probability that the next day is positive if the
current day changes by:
+2%, +3%, +4%, +5%, and +6%.
Example: if Apple rises 5% today, what is the historical probability that the next day
closes positive?
Use the last 2 years of historical data.
6. Repeat the same analysis for negative daily moves, such as:
-2%, -3%, -4%, and -5%.
7. Repeat both of the above analyses again using 5 years of historical data.
8. Create a comparison table for the five selected stocks.
Include a simple correlation / collinearity analysis between them.
9. Plot a comparison chart for two to five stocks using daily data for one year.
Example: compare Apple and Amazon.
Normalize or rescale prices if needed so they can be compared clearly.
10. Create another table for the same five stocks showing the probability that the next day
is positive after:
● 2 consecutive positive days
● 3 consecutive positive days
● 4 consecutive positive days
● 5 consecutive positive days
● 6 consecutive positive days
11. Repeat the same analysis for consecutive negative days.
12. Screen all Nasdaq and NYSE stocks and identify the top 10 trending stocks, ranked
by daily percentage change.
13. Repeat the screening, but only include stocks with a market capitalization of $5 billion
or more.
This may require obtaining market cap data from an API or web scraping source.

I will be using the Massive API, formerly polygon.io. Massive obtains its data from an aggregate of all major US stock exchanges. The plan I am selecting provides 5 years of historical data.

In [1]:
import requests
import pandas as pd
from pathlib import Path
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

# Bypass scientific notation
pd.options.display.float_format = '{:,.0f}'.format


def load_api_key(filepath="api_keys/massive.txt"):
    """Load Massive API key from a local text file."""
    return Path(filepath).read_text(encoding="utf-8").strip()


def get_all_pages(url, params=None):
    """Fetch all paginated results from Massive."""
    all_results = []

    while url:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        if data.get("status") == "NOT_AUTHORIZED":
            raise PermissionError(data.get("message", "Not authorized for this request."))

        results = data.get("results", [])
        all_results.extend(results)

        # After the first request, next_url already contains the needed query info
        url = data.get("next_url")
        params = None

    return all_results


def get_massive_daily_bars(
    ticker="AAPL",
    api_key_path="api_keys/massive.txt",
    years=5,
    safety_days=1,
    adjusted=True,
    sort="asc"
):
    """
    Fetch daily aggregate bars for a ticker from Massive.

    Parameters
    ----------
    ticker : str
        Stock ticker symbol, e.g. 'AAPL'
    api_key_path : str
        Path to text file containing API key
    years : int
        Number of years of history to request
    safety_days : int
        Number of days to move forward from exact cutoff to avoid entitlement edge issues
    adjusted : bool
        Whether to return adjusted prices
    sort : str
        'asc' or 'desc'

    Returns
    -------
    pd.DataFrame
    """
    api_key = load_api_key(api_key_path)

    today = date.today()
    start_date = today - relativedelta(years=years) + timedelta(days=safety_days)

    start_str = start_date.isoformat()
    end_str = today.isoformat()

    url = f"https://api.massive.com/v2/aggs/ticker/{ticker}/range/1/day/{start_str}/{end_str}"

    params = {
        "adjusted": str(adjusted).lower(),
        "sort": sort,
        "limit": 50000,
        "apiKey": api_key,
    }

    results = get_all_pages(url, params)

    if not results:
        raise ValueError(f"No data returned for {ticker}.")

    df = pd.DataFrame(results).rename(columns={
        "t": "timestamp",
        "o": "open",
        "h": "high",
        "l": "low",
        "c": "close",
        "v": "volume",
        "vw": "vwap",
        "n": "transactions",
    })

    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms").dt.date
    df["ticker"] = ticker

    preferred_order = [
        "timestamp", "ticker", "open", "high", "low", "close",
        "volume", "vwap", "transactions"
    ]
    df = df[[col for col in preferred_order if col in df.columns]]

    return df

def fetch_and_save_ticker(
    ticker,
    api_key_path="api_keys/massive.txt",
    years=5,
    safety_days=1,
    adjusted=True,
    sort="asc",
    save_folder="datasets"
):
    df = get_massive_daily_bars(
        ticker=ticker,
        api_key_path=api_key_path,
        years=years,
        safety_days=safety_days,
        adjusted=adjusted,
        sort=sort
    )

    filename = f"{save_folder}/{ticker.lower()}_daily.csv"
    df.to_csv(filename, index=False)

    print(f"{ticker} saved to {filename}")
    return df


tickers = [
    # Big Tech / Growth
    "AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "META", "TSLA",
    
    # Finance
    "JPM", "BAC", "GS", "MS",
    
    # Consumer / Retail
    "WMT", "COST", "HD", "NKE", "SBUX",
    
    # Healthcare / Pharma
    "JNJ", "PFE", "MRK", "UNH",
    
    # Energy
    "XOM", "CVX",
    
    # Industrials / Transportation
    "BA", "CAT", "GE", "UPS",
    
    # ETFs (nice for comparison)
    "SPY", "QQQ", "DIA"
]



# Maybe better structure, don't worry about it for now until I can do more testing
# tickers_by_sector = {
#     "Tech": ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "META"],
#     "Finance": ["JPM", "BAC", "GS", "MS"],
#     "Consumer": ["WMT", "COST", "HD", "NKE", "SBUX"],
#     "Healthcare": ["JNJ", "PFE", "MRK", "UNH"],
#     "Energy": ["XOM", "CVX"],
#     "Industrial": ["BA", "CAT", "GE", "UPS"],
#     "ETFs": ["SPY", "QQQ", "DIA"]
# }




data = {}

for t in tickers:
    data[t] = fetch_and_save_ticker(t)

AAPL saved to datasets/aapl_daily.csv
MSFT saved to datasets/msft_daily.csv
NVDA saved to datasets/nvda_daily.csv
GOOGL saved to datasets/googl_daily.csv
AMZN saved to datasets/amzn_daily.csv
META saved to datasets/meta_daily.csv
TSLA saved to datasets/tsla_daily.csv
JPM saved to datasets/jpm_daily.csv
BAC saved to datasets/bac_daily.csv
GS saved to datasets/gs_daily.csv
MS saved to datasets/ms_daily.csv
WMT saved to datasets/wmt_daily.csv
COST saved to datasets/cost_daily.csv
HD saved to datasets/hd_daily.csv
NKE saved to datasets/nke_daily.csv
SBUX saved to datasets/sbux_daily.csv
JNJ saved to datasets/jnj_daily.csv
PFE saved to datasets/pfe_daily.csv
MRK saved to datasets/mrk_daily.csv
UNH saved to datasets/unh_daily.csv
XOM saved to datasets/xom_daily.csv
CVX saved to datasets/cvx_daily.csv
BA saved to datasets/ba_daily.csv
CAT saved to datasets/cat_daily.csv
GE saved to datasets/ge_daily.csv
UPS saved to datasets/ups_daily.csv
SPY saved to datasets/spy_daily.csv
QQQ saved to datas

In [3]:
ticker_names = {
    # Big Tech / Growth
    "AAPL": "Apple Inc.",
    "MSFT": "Microsoft Corporation",
    "NVDA": "NVIDIA Corporation",
    "GOOGL": "Alphabet Inc. (Class A)",
    "AMZN": "Amazon.com, Inc.",
    "META": "Meta Platforms, Inc.",
    "TSLA": "Tesla, Inc.",
    
    # Finance
    "JPM": "JPMorgan Chase & Co.",
    "BAC": "Bank of America Corporation",
    "GS": "Goldman Sachs Group, Inc.",
    "MS": "Morgan Stanley",
    
    # Consumer / Retail
    "WMT": "Walmart Inc.",
    "COST": "Costco Wholesale Corporation",
    "HD": "The Home Depot, Inc.",
    "NKE": "NIKE, Inc.",
    "SBUX": "Starbucks Corporation",
    
    # Healthcare / Pharma
    "JNJ": "Johnson & Johnson",
    "PFE": "Pfizer Inc.",
    "MRK": "Merck & Co., Inc.",
    "UNH": "UnitedHealth Group Incorporated",
    
    # Energy
    "XOM": "Exxon Mobil Corporation",
    "CVX": "Chevron Corporation",
    
    # Industrials / Transportation
    "BA": "The Boeing Company",
    "CAT": "Caterpillar Inc.",
    "GE": "General Electric Company",
    "UPS": "United Parcel Service, Inc.",
    
    # ETFs
    "SPY": "SPDR S&P 500 ETF Trust",
    "QQQ": "Invesco QQQ Trust",
    "DIA": "SPDR Dow Jones Industrial Average ETF Trust"
}

# Daily master
Do I need to keep this? Could I just work with the five years I got straight out of the box?

In [2]:
from pathlib import Path
import pandas as pd

def update_daily_master(
    ticker,
    api_key_path="api_keys/massive.txt",
    years=5,
    safety_days=1,
    adjusted=True,
    sort="asc",
    save_folder="datasets"
):
    save_path = Path(save_folder)
    save_path.mkdir(parents=True, exist_ok=True)

    master_file = save_path / f"{ticker.lower()}_daily_master.csv"

    new_df = get_massive_daily_bars(
        ticker=ticker,
        api_key_path=api_key_path,
        years=years,
        safety_days=safety_days,
        adjusted=adjusted,
        sort=sort
    )

    if master_file.exists():
        old_df = pd.read_csv(master_file)
        old_df["timestamp"] = pd.to_datetime(old_df["timestamp"]).dt.date

        combined = pd.concat([old_df, new_df], ignore_index=True)
    else:
        combined = new_df.copy()

    combined["timestamp"] = pd.to_datetime(combined["timestamp"]).dt.date

    combined = (
        combined
        .drop_duplicates(subset=["timestamp"], keep="last")
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    combined.to_csv(master_file, index=False)

    print(f"{ticker} master updated: {master_file}")
    print(combined.head())
    print(combined.tail())
    print(combined.shape)
    print("Date range:", combined["timestamp"].min(), "to", combined["timestamp"].max())

    return combined

In [3]:
for ticker in tickers:
    data[ticker] = update_daily_master(ticker)

AAPL master updated: datasets/aapl_daily_master.csv
    timestamp ticker  open  high  low  close      volume  vwap  transactions
0  2021-04-22   AAPL   133   134  131    132  84,566,456   133        615670
1  2021-04-23   AAPL   132   135  132    134  78,666,779   134        519533
2  2021-04-26   AAPL   135   135  134    135  66,888,509   135        484069
3  2021-04-27   AAPL   135   135  134    134  66,015,804   135        480003
4  2021-04-28   AAPL   134   135  133    134 107,746,597   135        783355
       timestamp ticker  open  high  low  close     volume  vwap  transactions
1250  2026-04-15   AAPL   258   267  258    266 49,913,511   264        728560
1251  2026-04-16   AAPL   267   267  261    263 43,323,112   263        635080
1252  2026-04-17   AAPL   267   272  267    270 61,436,228   270        723488
1253  2026-04-20   AAPL   270   274  270    273 36,582,599   273        541032
1254  2026-04-21   AAPL   272   273  265    266 50,192,036   268        710075
(1255, 9)
Da

In [ ]:
aapl_daily = update_daily_master("AAPL")
msft_daily = update_daily_master("MSFT")
nvda_daily = update_daily_master("NVDA")